# 🖨️ VRM / FBX → 3D Print v41 (Full Notebook)

**DynaMesh方式ソリッド — SDF + Gaussianスムーズ + rtree fallback対応**

### solidモードの仕組み
```
VDBメッシュ → voxel化 → 内部充填
  → SDF（符号付き距離場）構築
  → Gaussianスムーズ（SDF空間で滑らかに）
  → isosurface抽出（level=0）
  = 滑らかで完全に中身が詰まったソリッド
```
加えて、`trimesh.proximity.closest_point` が `rtree` 未導入で失敗する環境向けにフォールバックを実装しています。


In [ ]:
import sys
!apt-get update -qq && apt-get install -y -qq libopengl0 libgl1-mesa-glx libglib2.0-0 libspatialindex-dev assimp-utils
!{sys.executable} -m pip install -q meshlib trimesh open3d numpy fast-simplification scikit-image scipy rtree
print("✅ 完了！ Kernel → Restart Kernel → 次のセルへ")

In [ ]:
import subprocess, os, tempfile, traceback, gc, time
import numpy as np
import trimesh
import open3d as o3d
import meshlib.mrmeshpy as mr
import meshlib.mrmeshnumpy as mrn

print("✅ インポート完了")


def load_submeshes(path, tmpdir):
    try:
        scene = trimesh.load(path, force="scene")
    except Exception:
        obj = os.path.join(tmpdir, "tmp.obj")
        subprocess.run(["assimp", "export", path, obj], check=True, capture_output=True)
        scene = trimesh.load(obj, force="scene")
    if isinstance(scene, trimesh.Scene):
        return [m for m in scene.geometry.values()
                if isinstance(m, trimesh.Trimesh) and len(m.faces) > 0]
    return [scene]


def auto_scale(meshes):
    verts = np.concatenate([m.vertices for m in meshes])
    sz = (verts.max(0) - verts.min(0)).max()
    if sz > 10:
        for m in meshes:
            m.vertices *= 0.001
        return meshes, f"⚠️  補正: {sz:.1f}mm→{sz*0.001:.3f}m"
    return meshes, f"✅ スケールOK: {sz:.4f}m"


def to_mr(m):
    return mrn.meshFromFacesVerts(
        np.array(m.faces, np.int32),
        np.array(m.vertices, np.float32),
    )


def from_mr(mr_mesh):
    v = mrn.getNumpyVerts(mr_mesh)
    f = mrn.getNumpyFaces(mr_mesh.topology)
    r = trimesh.Trimesh(vertices=v, faces=f, process=False)
    valid = np.isfinite(r.vertices[r.faces]).all(axis=(1, 2))
    r.update_faces(valid)
    r.remove_unreferenced_vertices()
    return r


def uniform_subdivide(mesh, iterations=2):
    for _ in range(iterations):
        v, f = trimesh.remesh.subdivide(mesh.vertices, mesh.faces)
        mesh = trimesh.Trimesh(vertices=v, faces=f, process=False)
    return mesh


def taubin_smooth(mesh, iters=50):
    try:
        edges = mesh.edges_sorted
        unique, counts = np.unique(edges, axis=0, return_counts=True)
        boundary_edges = unique[counts == 1]
        boundary_verts = np.unique(boundary_edges.flatten())
        boundary_pos = mesh.vertices[boundary_verts].copy()
        o3m = o3d.geometry.TriangleMesh(
            vertices=o3d.utility.Vector3dVector(mesh.vertices),
            triangles=o3d.utility.Vector3iVector(mesh.faces),
        )
        o3m = o3m.filter_smooth_taubin(
            number_of_iterations=iters,
            lambda_filter=0.5,
            mu=-0.53,
        )
        result = trimesh.Trimesh(
            vertices=np.asarray(o3m.vertices),
            faces=np.asarray(o3m.triangles),
            process=False,
        )
        result.vertices[boundary_verts] = boundary_pos
        return result
    except Exception:
        return mesh


def closest_point_with_fallback(surface_mesh, query_points):
    """厳密投影を試し、rtree不足時のみKDTree頂点近似へフォールバック。"""
    try:
        cp, dist, tri_id = trimesh.proximity.closest_point(surface_mesh, query_points)
        return cp, dist, tri_id, "exact"
    except ModuleNotFoundError as e:
        if "rtree" not in str(e):
            raise
        return project_vertices_with_kdtree(query_points, surface_mesh.vertices, chunk_size=250_000) + (None, "kdtree-vertex")


def make_projection_points(mesh, max_points=1_500_000, seed=42):
    """投影用参照点を軽量化して作成。巨大メッシュでの停止を防ぐ。"""
    verts = np.asarray(mesh.vertices)
    n = len(verts)
    if n <= max_points:
        return verts
    rng = np.random.default_rng(seed)
    idx = rng.choice(n, size=max_points, replace=False)
    return verts[idx]


def project_vertices_with_kdtree(query_points, reference_points, chunk_size=250_000):
    """KDTreeで最近傍点をチャンク処理。巨大配列でもメモリ破綻しにくい。"""
    from scipy.spatial import cKDTree

    tree = cKDTree(reference_points)
    q = np.asarray(query_points)
    closest = np.empty_like(q)
    distances = np.empty(len(q), dtype=np.float64)

    for i in range(0, len(q), chunk_size):
        j = min(i + chunk_size, len(q))
        dist, idx = tree.query(q[i:j], k=1)
        closest[i:j] = reference_points[idx]
        distances[i:j] = dist

    return closest, distances


def make_solid_dynmesh(mesh, pitch, L, sdf_smooth=1.0):
    """DynaMesh方式ソリッド化 — SDF + Gaussianスムーズ + isosurface"""
    from scipy.ndimage import binary_fill_holes, binary_dilation, distance_transform_edt, gaussian_filter
    from skimage.measure import marching_cubes

    L(f"🔄 DynaMeshソリッド化 (pitch={pitch*1000:.2f}mm)...")
    vox = mesh.voxelized(pitch)
    matrix = vox.matrix.copy()
    grid_size = matrix.shape[0] * matrix.shape[1] * matrix.shape[2]
    mem_gb = grid_size * 4 * 4 / 1e9
    L(f"  voxel化: {matrix.shape}  SDF推定メモリ: {mem_gb:.1f}GB")
    if mem_gb > 30:
        L("  ⚠️ メモリ不足の可能性！SOLID_VOXELを大きくしてください")

    dilated = binary_dilation(matrix, iterations=2)
    solid = binary_fill_holes(dilated)
    del dilated
    L(f"  内部充填: {solid.sum():,} voxels")

    L("🔄 SDF構築中...")
    dist_outside = distance_transform_edt(~solid).astype(np.float32)
    dist_inside = distance_transform_edt(solid).astype(np.float32)
    sdf = dist_outside - dist_inside
    del dist_outside, dist_inside, solid

    if sdf_smooth > 0:
        sdf = gaussian_filter(sdf, sigma=sdf_smooth)
        L(f"  SDFスムーズ (sigma={sdf_smooth})")

    L("🔄 isosurface抽出...")
    verts, faces, _, _ = marching_cubes(sdf, level=0.0)
    del sdf

    verts = verts * pitch + np.array(vox.translation)
    result = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
    L(f"  完了: {len(result.faces):,}面")
    return result


def watertight_fix(mr_mesh):
    p = mr.FillHoleParams()
    for e in mr_mesh.topology.findHoleRepresentiveEdges():
        mr.fillHole(mr_mesh, e, p)
    return mr_mesh


def run_pipeline(
    input_file,
    output_dir="/home",
    subdivide_iters=2,
    smooth_iters=50,
    voxel_size=0.0001,
    merge_offset=1.5,
    final_faces=20_000_000,
    output_mode="solid",
    solid_voxel=None,
    projection_blend=0.4,
    projection_mode="auto",   # "auto" / "exact" / "kdtree"
    projection_max_points=1_500_000,
    projection_chunk=250_000,
    shell_thickness=0.001,
    do_watertight=True,
    output_scale=1000.0,
):
    log = []
    final_path = None
    t0 = time.time()

    def L(msg):
        log.append(f"{msg}  [{time.time()-t0:.0f}s]")
        print(log[-1])

    try:
        with tempfile.TemporaryDirectory() as tmp:
            L("🔄 読み込み中...")
            meshes = load_submeshes(input_file, tmp)
            L(f"✅ {len(meshes)}個  面:{sum(len(m.faces) for m in meshes):,}")

            meshes, smsg = auto_scale(meshes)
            L(smsg)

            L(f"🔄 各パーツ: subdivide x{subdivide_iters} → Taubin x{smooth_iters}...")
            smoothed = []
            for i, m in enumerate(meshes):
                t_part = time.time()
                if len(m.faces) < 4:
                    smoothed.append(m)
                    continue
                m2 = m.copy()
                m2.merge_vertices(digits_vertex=14)
                m2.update_faces(m2.nondegenerate_faces())
                m2.update_faces(m2.unique_faces())
                m2.remove_unreferenced_vertices()
                m2.fix_normals()
                if len(m2.faces) < 4:
                    smoothed.append(m)
                    continue
                sub = uniform_subdivide(m2, subdivide_iters)
                sub = taubin_smooth(sub, smooth_iters)
                dt = time.time() - t_part
                L(f"  [{i+1}/{len(meshes)}] {len(m.faces):,}→{len(sub.faces):,}  {dt:.1f}s")
                smoothed.append(sub)

            L("🔄 全パーツ結合中...")
            mesh = trimesh.util.concatenate(smoothed)
            del smoothed
            gc.collect()
            L(f"✅ 結合: {len(mesh.faces):,}面")

            L(f"🔄 VDB (voxel={voxel_size:.5f}, offset={merge_offset})...")
            mr_mesh = to_mr(mesh)
            del mesh
            gc.collect()
            p = mr.GeneralOffsetParameters()
            p.voxelSize = float(voxel_size)
            mr_result = mr.generalOffsetMesh(mr_mesh, float(merge_offset) * float(voxel_size), p)
            del mr_mesh
            gc.collect()
            vdb_faces = mr_result.topology.numValidFaces()
            L(f"✅ VDB完了  {vdb_faces:,}面")

            tmp_stl = os.path.join(tmp, "tmp_vdb.stl")
            mr.saveMesh(mr_result, tmp_stl)
            del mr_result
            gc.collect()
            vdb_mesh = trimesh.load(tmp_stl)
            if isinstance(vdb_mesh, trimesh.Scene):
                vdb_mesh = trimesh.util.concatenate(list(vdb_mesh.geometry.values()))

            L("🔄 外殻抽出中...")
            components = vdb_mesh.split(only_watertight=False)
            if len(components) > 1:
                outer = max(components, key=lambda c: len(c.faces))
                L(f"  {len(components)}コンポーネント → 外殻のみ（{len(components)-1}個除去）")
            else:
                outer = vdb_mesh
            L(f"  外殻: {len(outer.faces):,}面")
            del vdb_mesh, components
            gc.collect()

            if output_mode == "solid":
                sv = solid_voxel if solid_voxel else voxel_size * 2
                solid_mesh = make_solid_dynmesh(outer, sv, L)

                L("🔄 Step1: Taubinスムーズ（ボクセル感除去）...")
                solid_mesh = taubin_smooth(solid_mesh, smooth_iters)
                L(f"  {len(solid_mesh.faces):,}面")

                L("🔄 Step2: VDB表面にprojection（細部復元）...")
                blend = projection_blend
                if blend <= 0:
                    L("  projectionスキップ (blend<=0)")
                else:
                    proj_mode = None
                    closest_pts = None

                    # exactは超巨大メッシュで非常に遅いので制限
                    allow_exact = (
                        projection_mode in ("auto", "exact")
                        and len(outer.faces) <= 2_000_000
                        and len(solid_mesh.vertices) <= 3_000_000
                    )
                    if allow_exact:
                        try:
                            closest_pts, distances, _, proj_mode = closest_point_with_fallback(outer, solid_mesh.vertices)
                        except Exception as e:
                            L(f"  exact projection失敗/回避 → KDTreeへフォールバック ({type(e).__name__})")

                    if closest_pts is None:
                        ref_points = make_projection_points(outer, max_points=int(projection_max_points))
                        closest_pts, distances = project_vertices_with_kdtree(
                            solid_mesh.vertices,
                            ref_points,
                            chunk_size=int(projection_chunk),
                        )
                        proj_mode = f"kdtree-points({len(ref_points):,})"

                    solid_mesh.vertices = solid_mesh.vertices * (1 - blend) + closest_pts * blend
                    L(f"  projection完了 (blend={blend}, mode={proj_mode})")

                L("🔄 Step3: 仕上げスムーズ...")
                solid_mesh = taubin_smooth(solid_mesh, max(10, smooth_iters // 5))
                L(f"  {len(solid_mesh.faces):,}面")

                del outer
                gc.collect()
                mr_result = to_mr(solid_mesh)
                del solid_mesh
                gc.collect()

                p_fill = mr.FillHoleParams()
                holes = mr_result.topology.findHoleRepresentiveEdges()
                hole_count = 0
                for e in holes:
                    mr.fillHole(mr_result, e, p_fill)
                    hole_count += 1
                if hole_count:
                    L(f"  穴埋め: {hole_count}個")
                L(f"✅ ソリッド完了  {mr_result.topology.numValidFaces():,}面")

            elif output_mode == "shell":
                L(f"🔄 シェル化（厚み {shell_thickness*1000:.2f}mm）...")
                mr_outer = to_mr(outer)
                p_inner = mr.GeneralOffsetParameters()
                p_inner.voxelSize = float(voxel_size)
                mr_inner = mr.generalOffsetMesh(mr_outer, -float(shell_thickness), p_inner)
                L(f"  内側: {mr_inner.topology.numValidFaces():,}面")
                try:
                    bool_result = mr.boolean(mr_outer, mr_inner, mr.BooleanOperation.DifferenceAB)
                    mr_result = bool_result.mesh
                    L(f"✅ シェル化完了（Boolean）  {mr_result.topology.numValidFaces():,}面")
                except Exception:
                    L("  Boolean失敗 → 反転結合で代替")
                    inner_m = from_mr(mr_inner)
                    inner_m.invert()
                    shell = trimesh.util.concatenate([outer, inner_m])
                    mr_result = to_mr(shell)
                    del inner_m, shell
                    L("✅ シェル化完了（結合）")
                del mr_outer, mr_inner
                gc.collect()

            else:
                mr_result = to_mr(outer)
                L(f"✅ rawモード  {mr_result.topology.numValidFaces():,}面")

            tmp2 = os.path.join(tmp, "tmp_mode.stl")
            mr.saveMesh(mr_result, tmp2)
            del mr_result
            gc.collect()
            mr_result = mr.loadMesh(tmp2)
            cur = mr_result.topology.numValidFaces()
            tgt = int(final_faces)
            if cur > tgt:
                L(f"🔄 デシメーション {cur:,} → {tgt:,}面...")
                s = mr.DecimateSettings()
                s.maxDeletedFaces = cur - tgt
                mr.decimateMesh(mr_result, s)
                L(f"✅ {mr_result.topology.numValidFaces():,}面")

            if do_watertight:
                L("🔄 水密修復...")
                mr_result = watertight_fix(mr_result)
                L("✅ 完了")

            tmp_final = os.path.join(tmp, "tmp_final.stl")
            mr.saveMesh(mr_result, tmp_final)
            del mr_result
            gc.collect()

            result_mesh = trimesh.load(tmp_final)
            if isinstance(result_mesh, trimesh.Scene):
                result_mesh = trimesh.util.concatenate(list(result_mesh.geometry.values()))

            if abs(output_scale - 1.0) > 0.001:
                result_mesh.vertices *= output_scale
                L(f"📏 出力スケール x{output_scale}")

            src_name = os.path.splitext(os.path.basename(input_file))[0]
            final_path = os.path.join(output_dir, f"{src_name}_{output_mode}.stl")
            result_mesh.export(final_path)

            mb = os.path.getsize(final_path) / 1024 / 1024
            L(
                f"\n🎉 完成！  {final_path}\n   面数:{len(result_mesh.faces):,}  {mb:.1f}MB  水密:{result_mesh.is_watertight}\n   モード: {output_mode}"
            )
            del result_mesh
            gc.collect()

    except Exception as e:
        log.append(f"\n❌ {e}\n{traceback.format_exc()}")
        print(log[-1])

    return "\n".join(log), final_path


print("✅ 準備完了")


In [ ]:
# ═══════════════════════════════════════════════════
#  パラメータ一覧
# ═══════════════════════════════════════════════════
INPUT_FILE        = "/home/青威リューガ_24cm.fbx"
OUTPUT_DIR        = "/home"

# ── サブディビジョン＋スムーズ ──
SUBDIVIDE_ITERS   = 2
SMOOTH_ITERS      = 50

# ── VDB ──
VOXEL             = 0.0001
MERGE_OFFSET      = 1.5

# ── 出力モード ──
OUTPUT_MODE       = "solid"  # "solid" / "shell" / "raw"
SOLID_VOXEL       = 0.0003
PROJECTION_BLEND  = 0.25
PROJECTION_MODE   = "kdtree"   # 重いモデルは"kdtree"推奨
PROJECTION_MAX_POINTS = 1_500_000
PROJECTION_CHUNK  = 200_000
SHELL_THICKNESS   = 0.001

# ── 仕上げ ──
FINAL_FACES       = 20_000_000
WATERTIGHT        = True
OUTPUT_SCALE      = 1000.0

# ═══════════════════════════════════════════════════
log, out = run_pipeline(
    INPUT_FILE, OUTPUT_DIR,
    subdivide_iters=SUBDIVIDE_ITERS,
    smooth_iters=SMOOTH_ITERS,
    voxel_size=VOXEL,
    merge_offset=MERGE_OFFSET,
    final_faces=FINAL_FACES,
    output_mode=OUTPUT_MODE,
    solid_voxel=SOLID_VOXEL,
    projection_blend=PROJECTION_BLEND,
    projection_mode=PROJECTION_MODE,
    projection_max_points=PROJECTION_MAX_POINTS,
    projection_chunk=PROJECTION_CHUNK,
    shell_thickness=SHELL_THICKNESS,
    do_watertight=WATERTIGHT,
    output_scale=OUTPUT_SCALE,
)
if out:
    print(f"\n✅ 出力: {out}")
